# Week 3 — Retrieval Pipeline: MSA vs. Darija Query Mismatch

Measures Recall@k and MRR for three conditions against the same MSA corpus:
- **(a) BASELINE** — retrieve using the MSA query
- **(b) MISMATCH** — retrieve using the Darija query
- **(c) MITIGATION** — normalize Darija → MSA (via LLM), then retrieve

Runs entirely on free Colab: hybrid BM25 + dense embeddings in FAISS, no local GPU required. Loads the dataset directly from the [GitHub repo](https://github.com/Rania-khaoudane/MSA). Needs a `GROQ_API_KEY` in Colab Secrets for the mitigation condition (same setup as the validation notebook).

### Install dependencies (Colab only, run once)

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers groq requests

### Load the dataset directly from the GitHub repo

In [ ]:
import json, requests

# -----------------------------------------------------------------------
# CORPUS_MODE controls what the 300 pilot questions are retrieved against.
#
#   "pilot"  -> the original 80 author-written passages (as first built)
#   "large"  -> the same 300 questions, but searched against 3,054 passages
#               (80 pilot + 2,974 real Arabic Wikipedia). Same questions,
#               same gold answers; only the pool of distractors grows.
#
# "large" is the stronger setup: retrieval has to find the right passage
# among thousands of real ones rather than 79 author-written alternatives,
# which removes the objection that the corpus and questions were written
# together. It needs corpus_v2.json uploaded to the session.
# -----------------------------------------------------------------------
CORPUS_MODE = "large"        # "pilot" or "large"

REPO_RAW = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main"
qa_pairs = json.loads(requests.get(f"{REPO_RAW}/data/qa_pairs.json").text)

if CORPUS_MODE == "large":
    with open("corpus_v2.json", encoding="utf-8") as f:
        corpus = json.load(f)
else:
    corpus = json.loads(requests.get(f"{REPO_RAW}/data/corpus.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]

# Sanity check: every question's gold passage must exist in the chosen corpus,
# otherwise Recall is silently capped below 100%.
gold_ids = {q["source_chunk_id"] for q in qa_pairs}
missing = gold_ids - set(corpus_ids)

print(f"Mode: {CORPUS_MODE}")
print(f"Corpus: {len(corpus)} passages | QA pairs: {len(qa_pairs)}")
print(f"Unresolved gold passages: {len(missing)}" + (f" -> {sorted(missing)[:5]}" if missing else ""))

### Build the dense embedding index (FAISS)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

corpus_embeddings = embedder.encode(
    corpus_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64
).astype("float32")

dim = corpus_embeddings.shape[1]
dense_index = faiss.IndexFlatIP(dim)
dense_index.add(corpus_embeddings)

print(f"FAISS index built: {dense_index.ntotal} passages, dim={dim}")

### Build the sparse lexical index (BM25)

In [ ]:
import re
from rank_bm25 import BM25Okapi

ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]')

def normalize_arabic(text: str) -> str:
    """
    Light Arabic orthographic normalization to reduce spurious lexical
    mismatches in BM25 (different alef forms, taa marbuta vs haa,
    diacritics, tatweel) -- this is the orthographic-normalization
    preprocessing step named in the project's own method section, which
    the first version of this pipeline had skipped.
    """
    text = ARABIC_DIACRITICS.sub('', text)
    text = re.sub(r'[\u0625\u0623\u0622\u0627]', '\u0627', text)  # unify alef forms -> ا
    text = re.sub(r'\u0649', '\u064A', text)                         # alef maqsura -> yaa
    text = re.sub(r'\u0629', '\u0647', text)                         # taa marbuta -> haa
    text = re.sub(r'\u0624', '\u0648', text)                         # waw hamza -> waw
    text = re.sub(r'\u0626', '\u064A', text)                         # yaa hamza -> yaa
    text = re.sub(r'\u0640+', '', text)                               # remove tatweel
    text = re.sub(r'[^\w\s]', ' ', text)                              # strip punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def simple_tokenize(text: str):
    return normalize_arabic(text).split()

tokenized_corpus = [simple_tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built (with Arabic orthographic normalization applied to tokenization).")

### Hybrid retrieval function

In [ ]:
ALPHA = 0.6  # dense weight; slightly favors dense embeddings over BM25 by default,
             # since dense handles paraphrase/dialectal wording better than lexical overlap.
             # Re-tuned empirically in the next cell.

def retrieve(query: str, k: int = 5, alpha: float = None):
    """
    Hybrid retrieval: alpha weights dense score, (1-alpha) weights BM25 score.
    Both are min-max normalized to [0,1] before combining, since they're on
    different scales (cosine similarity vs. BM25 term-weighting scores).
    Returns a list of (chunk_id, combined_score) of length k, best first.
    """
    if alpha is None:
        alpha = ALPHA

    q_emb = embedder.encode([query], normalize_embeddings=True)
    dense_scores = (corpus_embeddings @ q_emb[0])

    bm25_scores = np.array(bm25.get_scores(simple_tokenize(query)))

    def normalize(arr):
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)

    dense_norm = normalize(dense_scores)
    bm25_norm = normalize(bm25_scores)

    combined = alpha * dense_norm + (1 - alpha) * bm25_norm
    top_k_idx = np.argsort(-combined)[:k]

    return [(corpus_ids[i], float(combined[i])) for i in top_k_idx]

sample = retrieve(qa_pairs[0]["msa_query"], k=5)
print("Sample query:", qa_pairs[0]["msa_query"])
print("Expected chunk:", qa_pairs[0]["source_chunk_id"])
print("Top-5 retrieved:", sample)

### Recall@k and MRR evaluation function

In [ ]:
def evaluate(query_field: str, k_values=(1, 3, 5), max_k=5):
    """
    query_field: which field to use as the query ('msa_query', 'darija_query',
                 or a custom list aligned with qa_pairs for the mitigation condition).
    Returns a dict of {Recall@k: value, ..., MRR: value}.
    """
    hits_at_k = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for item in qa_pairs:
        query = item[query_field] if isinstance(query_field, str) else query_field[item["id"]]
        results = retrieve(query, k=max_k)
        retrieved_ids = [r[0] for r in results]
        gold_id = item["source_chunk_id"]

        # Recall@k: was the gold chunk in the top-k?
        for k in k_values:
            if gold_id in retrieved_ids[:k]:
                hits_at_k[k] += 1

        # MRR: reciprocal rank of the gold chunk (0 if not found in top max_k)
        if gold_id in retrieved_ids:
            rank = retrieved_ids.index(gold_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    n = len(qa_pairs)
    results_dict = {f"Recall@{k}": hits_at_k[k] / n for k in k_values}
    results_dict["MRR"] = sum(reciprocal_ranks) / n
    return results_dict

### Tune ALPHA (dense vs. BM25 weight) empirically on the mismatch condition

In [ ]:
import pandas as pd

# Empirically tune ALPHA (dense vs. BM25 weight) using the MISMATCH (Darija)
# condition -- this is the harder case where getting the balance right matters
# most, since dense embeddings tolerate paraphrase/dialectal wording much
# better than lexical (BM25) overlap does.

candidate_alphas = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
tuning_rows = []

for a in candidate_alphas:
    globals()["ALPHA"] = a
    mismatch_r5 = evaluate("darija_query", k_values=(5,))["Recall@5"]
    baseline_r5 = evaluate("msa_query", k_values=(5,))["Recall@5"]
    tuning_rows.append({"alpha": a, "Mismatch Recall@5": mismatch_r5, "Baseline Recall@5": baseline_r5})

tuning_df = pd.DataFrame(tuning_rows)
print(tuning_df)

best_row = tuning_df.loc[tuning_df["Mismatch Recall@5"].idxmax()]
ALPHA = float(best_row["alpha"])
print(f"\nBest ALPHA for the mismatch condition: {ALPHA} "
      f"(Mismatch Recall@5={best_row['Mismatch Recall@5']:.3f}, "
      f"Baseline Recall@5={best_row['Baseline Recall@5']:.3f})")
print("This ALPHA will now be used for all three conditions below, for a fair comparison.")

### Condition (a): BASELINE — MSA query

In [ ]:
print("Evaluating BASELINE (MSA query)...")
baseline_results = evaluate("msa_query")
print(baseline_results)

### Condition (b): MISMATCH — Darija query

In [ ]:
print("Evaluating MISMATCH (Darija query)...")
mismatch_results = evaluate("darija_query")
print(mismatch_results)

### Condition (c): MITIGATION — normalize Darija -> MSA, then retrieve

In [ ]:
# Condition (c): normalize Darija -> MSA, then retrieve.
# Requires GROQ_API_KEY in Colab Secrets (key icon, left sidebar).

import time
from groq import Groq, RateLimitError
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY not found in Colab Secrets.")
client = Groq(api_key=GROQ_API_KEY)

def normalize_to_msa(darija_text, model="openai/gpt-oss-20b", max_retries=5):
    """Rewrite a Darija question in MSA. Falls back to the original text on
    failure or empty output, so retrieval never runs on a blank query."""
    prompt = (
        'Rewrite Moroccan Darija questions into formal Modern Standard Arabic (Fusha), '
        'in the register used in encyclopedic writing. Preserve the exact meaning. '
        'Output ONLY the rewritten question.\n\n'
        'Example:\nDarija: "شحال ديال الساكنة كاينة فالمغرب؟"\nMSA: كم عدد سكان المغرب؟\n\n'
        f'Darija: "{darija_text}"\nMSA:'
    )
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=500,          # reasoning models need headroom or return empty
                reasoning_effort="low",  # keeps the reasoning budget small
            )
            out = (r.choices[0].message.content or "").strip().strip('"')
            return out if out else darija_text
        except RateLimitError:
            wait = 20 * (attempt + 1)
            print(f"  Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"  Normalization failed ({e}); using original.")
            return darija_text
    return darija_text

print(f"Normalizing {len(qa_pairs)} Darija queries...")
normalized_queries = {}
for i, item in enumerate(qa_pairs):
    normalized_queries[item["id"]] = normalize_to_msa(item["darija_query"])
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(qa_pairs)}")

with open("normalized_queries.json", "w", encoding="utf-8") as f:
    json.dump(normalized_queries, f, ensure_ascii=False, indent=2)

unchanged = sum(1 for it in qa_pairs if normalized_queries[it["id"]] == it["darija_query"])
print(f"\nDone. Unchanged (fallback) queries: {unchanged}/{len(qa_pairs)}")
print("Example:")
print("  Darija:     ", qa_pairs[0]["darija_query"])
print("  Normalized: ", normalized_queries[qa_pairs[0]["id"]])

### Evaluate MITIGATION condition

In [ ]:
print("Evaluating MITIGATION (normalized Darija -> MSA query)...")
mitigation_results = evaluate(normalized_queries)
print(mitigation_results)

### Final comparison table and headline numbers

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Baseline (MSA)":          baseline_results,
    "Mismatch (Darija)":       mismatch_results,
    "Mitigation (Normalized)": mitigation_results,
}).T

print(f"CORPUS_MODE = {CORPUS_MODE}   |   corpus = {len(corpus)} passages   |   n = {len(qa_pairs)} questions")
print("=" * 78)
print(comparison.to_string())
print("=" * 78)

for metric in [c for c in comparison.columns if c.startswith("Recall")] + ["MRR"]:
    b = baseline_results[metric]
    m = mismatch_results[metric]
    g = mitigation_results[metric]
    drop = b - m
    rec = ((g - m) / drop * 100) if drop > 0 else float("nan")
    print(f"  {metric:<10} drop {drop*100:5.1f} pts ({b*100:5.1f} -> {m*100:5.1f})   "
          f"normalization recovers {rec:6.1f}%")

comparison.to_csv(f"results_{CORPUS_MODE}.csv")
print(f"\nSaved results_{CORPUS_MODE}.csv")
print("\nRun this notebook once with CORPUS_MODE='pilot' and once with 'large',")
print("then compare the two CSVs.")